# Hat Kapasite — Ön-hesap JSON Üretici

**Amaç:** `hat_kapasite.json` üretmek. Panel startup'ta SQL sorgusu yapmak yerine doğrudan bu JSON'u okur.

**Kaynak:** `iett_data.db` → sefer tablosu (~10M satır) → her HATKODU için AVG(KAPASITE)

**Çıktı:**
- `panel_data/hat_kapasite.json` (Datathon)
- `iett_panel/panel_data/hat_kapasite.json` (panel canlı)

**Şema:** `{ "HATKODU": ortalama_kapasite_int, ... }`

**Kullanım yeri:** `services.py` → `gecikme_skoru` endpoint'i fallback'i (yoksa default 90).

**Not:** Sefer verisi değişirse bu notebook yeniden çalıştırılmalı.

In [1]:
import sqlite3
import json
import shutil
from pathlib import Path

DATATHON_DIR = Path(r'c:\Users\asus\Desktop\Datathon')
PANEL_DIR    = DATATHON_DIR / 'panel_data'
DB_PATH      = PANEL_DIR / 'iett_data.db'

conn = sqlite3.connect(DB_PATH)
print(f'DB: {DB_PATH}')

DB: c:\Users\asus\Desktop\Datathon\panel_data\iett_data.db


In [2]:
# Sefer tablosundan hat başına ortalama kapasite
cur = conn.cursor()
cur.execute("""
    SELECT HATKODU,
           ROUND(AVG(KAPASITE), 0) AS ort_kapasite,
           COUNT(*)                AS sefer_sayisi
    FROM sefer
    WHERE KAPASITE IS NOT NULL
      AND KAPASITE > 0
      AND HATKODU IS NOT NULL
      AND HATKODU != ''
    GROUP BY HATKODU
""")
satirlar = cur.fetchall()
print(f'Sorgu tamamlandi: {len(satirlar):,} hat')

# Liste → dict
hat_kapasite = {r[0]: int(r[1]) for r in satirlar if r[0] and r[1]}
print(f'Geçerli kayıt: {len(hat_kapasite):,} hat')

Sorgu tamamlandi: 826 hat
Geçerli kayıt: 826 hat


In [3]:
# İstatistik özet
kapasiteler = list(hat_kapasite.values())
print(f'Min kapasite:  {min(kapasiteler)}')
print(f'Max kapasite:  {max(kapasiteler)}')
print(f'Ortalama:      {sum(kapasiteler)/len(kapasiteler):.1f}')
print(f'Medyan:        {sorted(kapasiteler)[len(kapasiteler)//2]}')
print()
print('Örnek 5 hat:')
for k, v in list(hat_kapasite.items())[:5]:
    print(f'  {k}: {v}')

Min kapasite:  41
Max kapasite:  172
Ortalama:      92.5
Medyan:        94

Örnek 5 hat:
  1: 82
  10: 99
  10A: 136
  10B: 82
  10E: 109


In [4]:
# JSON çıktı
OUT_D = PANEL_DIR / 'hat_kapasite.json'
OUT_P = Path(r'c:\Users\asus\Desktop\iett_panel\panel_data\hat_kapasite.json')

with open(OUT_D, 'w', encoding='utf-8') as f:
    json.dump(hat_kapasite, f, ensure_ascii=False, indent=2)
print(f'Datathon: {OUT_D} ({OUT_D.stat().st_size/1024:.0f} KB)')

if OUT_P.parent.exists():
    shutil.copy2(OUT_D, OUT_P)
    print(f'Panel:    {OUT_P}')

conn.close()
print('Tamamlandı.')

Datathon: c:\Users\asus\Desktop\Datathon\panel_data\hat_kapasite.json (12 KB)
Panel:    c:\Users\asus\Desktop\iett_panel\panel_data\hat_kapasite.json
Tamamlandı.
